# Outfield P4 or not Models — Train V2

Split the **hitter** CSV into just the outfielders to do model development on them.

Building out the p4 or not models for the outfielders

**Source:** `backend/ml/train/train_v2/hitters/hitters_cleaned.csv`

In [29]:
import pandas as pd
import numpy as np
import re
import os
from pathlib import Path

HITTER_CSV = Path("hitters_cleaned.csv")

hitters = pd.read_csv(HITTER_CSV, low_memory=False)
print(f"Loaded {len(hitters):,} rows, {len(hitters.columns)} columns")

ofs = hitters[hitters["resolved_position"] == "OF"]
ofs.head()


Loaded 32,999 rows, 50 columns


,name,link,player_state,high_school,class,resolved_position,positions,commitment,commitment_date,height,...,of_velo_date,c_velo,c_velo_date,pop_time,pop_time_date,player_region,conference,division,college_location,committment_group
5,Nasir Akil,https://www.prepbaseballreport.com/profiles/CA...,CA,CAMPBELL HALL SCHOOL,2027,OF,OF,St. Mary's (9/10/25),NaN,74.0,...,1/05/25,NaN,NaN,NaN,NaN,West,West Coast,NCAA I,"MORAGA, CA",Non P4 D1
7,Cal Albright,https://www.prepbaseballreport.com/profiles/NY...,NY,Cortland,2027,OF,"OF,LHP",Hofstra,9/23/25,75.0,...,2/28/26,NaN,NaN,NaN,NaN,Northeast,Colonial,NCAA I,"HEMPSTEAD, NY",Non P4 D1
10,Elijah Alvarez,https://www.prepbaseballreport.com/profiles/CA...,CA,Norco,2027,OF,"OF,RHP",Cal State Northridge,11/17/25,72.0,...,7/30/25,NaN,NaN,NaN,NaN,West,Big West,NCAA I,"NORTHRIDGE, CA",Non P4 D1
13,Manning Amorde,https://www.prepbaseballreport.com/profiles/AL...,AL,UMS-Wright,2027,OF,"OF,LHP",South Alabama,8/10/25,73.0,...,1/11/26,NaN,NaN,NaN,NaN,South,Sun Belt,NCAA I,"MOBILE, AL",Non P4 D1
17,Chase Austin,https://www.prepbaseballreport.com/profiles/OR...,OR,Beaverton,2027,OF,"OF,LHP",Oregon State,8/27/25,73.0,...,2/15/25,NaN,NaN,NaN,NaN,West,Pacific 12,NCAA I,"BEAVERTON, OR",Non P4 D1


In [30]:
OF_MODELING_COLS = [
    # Identity / bio
    "player_state", "resolved_position",
    "height", "weight", "throwing_hand", "hitting_handedness",
    # Hitting stats (value + date for each)
    "exit_velo_max", "exit_velo_max_date",
    "exit_velo_avg", "exit_velo_avg_date",
    "distance_max", "distance_max_date",
    "sweet_spot_p", "sweet_spot_p_date",
    "hand_speed_max", "hand_speed_max_date",
    "bat_speed_max", "bat_speed_max_date",
    "rot_acc_max", "rot_acc_max_date",
    "hard_hit_p", "hard_hit_p_date",
    # Running and Defense (value + date)
    "sixty_time", "sixty_time_date",
    "thirty_time", "thirty_time_date",
    "ten_yard_time", "ten_yard_time_date",
    "run_speed_max", "run_speed_max_date",
    "of_velo", "of_velo_date",
    # Classification
    "player_region", "committment_group", "commitment_date"
]

OF_MODELING_COLS = [c for c in OF_MODELING_COLS if c in ofs.columns]
of_modeling = ofs[OF_MODELING_COLS]

# unknown is Non D1, with random juco/naia or canadian colleges or malformed urls for schools.
of_modeling = of_modeling[of_modeling["committment_group"] != "Unknown"] # removing like 600 rows
of_modeling = of_modeling[of_modeling["committment_group"] != "Non D1"]  # training on only d1 outfielders

print(of_modeling['committment_group'].value_counts())
print(of_modeling.shape)


committment_group
Non P4 D1    1727
P4            490
Name: count, dtype: int64
(2217, 35)


In [31]:
# new target feature for d1 or not modeling
of_modeling['p4_or_not'] = (of_modeling['committment_group'] != 'Non P4 D1').astype(int)

of_modeling['p4_or_not'].value_counts()

p4_or_not
0    1727
1     490
Name: count, dtype: int64

In [32]:
# ============================================================
# STALE DATA ANALYSIS
# ============================================================
# ---- EDITABLE PARAMETERS (change & re-run just this cell) ----
STALE_MONTHS = 24            # Stats older than this many months before graduation = "stale"
OUTLIER_STD_DEV = 2        # ± this many std devs from commitment-group mean = "outlier"
# ---------------------------------------------------------------

# --- Fresh copy so of_modeling stays untouched ---
of_model_recent = of_modeling.copy()

# --- Bring in class year (not in OF_MODELING_COLS) ---
of_model_recent["class"] = ofs.loc[of_model_recent.index, "class"]

# --- Stat <-> date column pairs (only OF-relevant stats) ---
STAT_DATE_PAIRS = [
    ("exit_velo_max",  "exit_velo_max_date"),
    ("exit_velo_avg",  "exit_velo_avg_date"),
    ("distance_max",   "distance_max_date"),
    ("sweet_spot_p",   "sweet_spot_p_date"),
    ("hand_speed_max", "hand_speed_max_date"),
    ("bat_speed_max",  "bat_speed_max_date"),
    ("rot_acc_max",    "rot_acc_max_date"),
    ("hard_hit_p",     "hard_hit_p_date"),
    ("sixty_time",     "sixty_time_date"),
    ("thirty_time",    "thirty_time_date"),
    ("ten_yard_time",  "ten_yard_time_date"),
    ("run_speed_max",  "run_speed_max_date"),
    ("of_velo",        "of_velo_date"),
]
# Filter to columns that actually exist
STAT_DATE_PAIRS = [(s, d) for s, d in STAT_DATE_PAIRS if s in of_model_recent.columns and d in of_model_recent.columns]

# --- Parse dates & compute months before graduation ---
def parse_pbr_date(d):
    """Parse PBR date format M/DD/YY to datetime."""
    if pd.isna(d) or str(d).strip() == "":
        return pd.NaT
    try:
        return pd.to_datetime(str(d).strip(), format="%m/%d/%y")
    except Exception:
        try:
            return pd.to_datetime(str(d).strip())
        except Exception:
            return pd.NaT

# Graduation reference = June 1 of class year
of_model_recent["grad_date"] = pd.to_datetime(of_model_recent["class"].astype(str) + "-06-01")

for stat_col, date_col in STAT_DATE_PAIRS:
    parsed_col = f"_{date_col}_parsed"
    months_col = f"{stat_col}__months_before_grad"
    of_model_recent[parsed_col] = of_model_recent[date_col].apply(parse_pbr_date)
    # Positive = stat recorded BEFORE graduation, negative = after graduation
    of_model_recent[months_col] = ((of_model_recent["grad_date"] - of_model_recent[parsed_col]).dt.days / 30.44).round(1)

print(f"Parsed {len(STAT_DATE_PAIRS)} stat date columns.\n")

# ============================================================
# 1. PER-STAT STALENESS BREAKDOWN
# ============================================================
print(f"{'='*70}")
print(f"1. PER-STAT STALENESS  (stale = >{STALE_MONTHS} months before graduation)")
print(f"{'='*70}\n")

rows = []
for stat_col, date_col in STAT_DATE_PAIRS:
    months_col = f"{stat_col}__months_before_grad"
    has_val  = of_model_recent[stat_col].notna()
    has_date = of_model_recent[months_col].notna()
    is_stale = of_model_recent[months_col] > STALE_MONTHS

    n_val   = has_val.sum()
    n_dated = (has_val & has_date).sum()
    n_stale = (has_val & is_stale).sum()
    n_no_date = n_val - n_dated

    rows.append({
        "stat": stat_col,
        "has_value": n_val,
        "has_date": n_dated,
        "no_date": n_no_date,
        "stale": n_stale,
        "stale_pct": round(100 * n_stale / n_val, 1) if n_val else 0,
    })

stale_df = pd.DataFrame(rows)
print(stale_df.to_string(index=False))

total_vals  = stale_df["has_value"].sum()
total_stale = stale_df["stale"].sum()
print(f"\nOverall: {total_stale:,} / {total_vals:,} stat values are stale ({100*total_stale/total_vals:.1f}%)")

# ============================================================
# 2. SENSITIVITY — staleness at different month thresholds
# ============================================================
print(f"\n{'='*70}")
print(f"2. STALENESS AT DIFFERENT THRESHOLDS")
print(f"{'='*70}\n")

thresholds = [12, 18, 24, 30, 36, 48]
for thresh in thresholds:
    t_stale = 0
    t_total = 0
    for stat_col, _ in STAT_DATE_PAIRS:
        months_col = f"{stat_col}__months_before_grad"
        has_val  = of_model_recent[stat_col].notna()
        is_stale = of_model_recent[months_col] > thresh
        t_stale += (has_val & is_stale).sum()
        t_total += has_val.sum()
    marker = "  <-- current" if thresh == STALE_MONTHS else ""
    print(f"  {thresh:>2} months: {t_stale:>5,} / {t_total:>6,} values stale ({100*t_stale/t_total:.1f}%){marker}")

# ============================================================
# 3. STALENESS BY COMMITMENT GROUP
# ============================================================
print(f"\n{'='*70}")
print(f"3. STALENESS BY COMMITMENT GROUP  (threshold = {STALE_MONTHS} months)")
print(f"{'='*70}\n")

for group in ["P4", "Non P4 D1", "Non D1"]:
    mask = of_model_recent["committment_group"] == group
    g_stale = 0
    g_total = 0
    for stat_col, _ in STAT_DATE_PAIRS:
        months_col = f"{stat_col}__months_before_grad"
        has_val  = of_model_recent.loc[mask, stat_col].notna()
        is_stale = of_model_recent.loc[mask, months_col] > STALE_MONTHS
        g_stale += (has_val & is_stale).sum()
        g_total += has_val.sum()
    pct = 100 * g_stale / g_total if g_total else 0
    n_players = mask.sum()
    print(f"  {group:>12}  ({n_players:>5,} players): {g_stale:>4,} / {g_total:>5,} values stale ({pct:.1f}%)")

# ============================================================
# 4. PLAYERS WITH ANY STALE STAT
# ============================================================
print(f"\n{'='*70}")
print(f"4. PLAYERS WITH ≥1 STALE STAT  (threshold = {STALE_MONTHS} months)")
print(f"{'='*70}\n")

# For each player, count how many of their stats are stale
stale_counts_per_player = pd.Series(0, index=of_model_recent.index)
stat_counts_per_player  = pd.Series(0, index=of_model_recent.index)

for stat_col, _ in STAT_DATE_PAIRS:
    months_col = f"{stat_col}__months_before_grad"
    has_val  = of_model_recent[stat_col].notna()
    is_stale = of_model_recent[months_col] > STALE_MONTHS
    stale_counts_per_player += (has_val & is_stale).astype(int)
    stat_counts_per_player  += has_val.astype(int)

of_model_recent["_n_stale_stats"] = stale_counts_per_player
of_model_recent["_n_total_stats"] = stat_counts_per_player

has_any_stale = stale_counts_per_player > 0
for group in ["P4", "Non P4 D1", "Non D1"]:
    mask = of_model_recent["committment_group"] == group
    n_total = mask.sum()
    n_affected = (mask & has_any_stale).sum()
    print(f"  {group:>12}: {n_affected:>4,} / {n_total:>5,} players have ≥1 stale stat ({100*n_affected/n_total:.1f}%)")

total_affected = has_any_stale.sum()
print(f"  {'Total':>12}: {total_affected:>4,} / {len(of_model_recent):>5,} players ({100*total_affected/len(of_model_recent):.1f}%)")

# ============================================================
# 5. OUTLIER + STALENESS INTERSECTION
# ============================================================
print(f"\n{'='*70}")
print(f"5. OUTLIER ANALYSIS  (±{OUTLIER_STD_DEV} std dev from group mean)")
print(f"   → How many outliers have stale stats?")
print(f"{'='*70}\n")

stat_value_cols = [s for s, _ in STAT_DATE_PAIRS]
outlier_rows = []

for stat_col in stat_value_cols:
    months_col = f"{stat_col}__months_before_grad"
    for group in ["P4", "Non P4 D1", "Non D1"]:
        mask = of_model_recent["committment_group"] == group
        vals = of_model_recent.loc[mask, stat_col]
        mean = vals.mean()
        std  = vals.std()

        if pd.isna(mean) or pd.isna(std) or std == 0:
            continue

        is_outlier = (vals < mean - OUTLIER_STD_DEV * std) | (vals > mean + OUTLIER_STD_DEV * std)
        n_outlier  = is_outlier.sum()
        n_with_val = vals.notna().sum()

        # Of those outliers, how many are stale?
        outlier_stale = (is_outlier & (of_model_recent.loc[mask, months_col] > STALE_MONTHS)).sum()
        # And how many are fresh?
        outlier_fresh = n_outlier - outlier_stale

        outlier_rows.append({
            "stat": stat_col, "group": group,
            "n": n_with_val, "outliers": n_outlier,
            "outlier_pct": round(100 * n_outlier / n_with_val, 1) if n_with_val else 0,
            "stale_outliers": outlier_stale,
            "fresh_outliers": outlier_fresh,
            "pct_outliers_stale": round(100 * outlier_stale / n_outlier, 1) if n_outlier else 0,
        })

outlier_df = pd.DataFrame(outlier_rows)

# Summary table: aggregate across stats per group
print("Aggregated across all stats per commitment group:\n")
for group in ["P4", "Non P4 D1", "Non D1"]:
    sub = outlier_df[outlier_df["group"] == group]
    tot_outliers = sub["outliers"].sum()
    tot_stale_o  = sub["stale_outliers"].sum()
    tot_fresh_o  = sub["fresh_outliers"].sum()
    pct = 100 * tot_stale_o / tot_outliers if tot_outliers else 0
    print(f"  {group:>12}: {tot_outliers:>4} outliers total → {tot_stale_o:>3} stale ({pct:.0f}%), {tot_fresh_o:>3} fresh ({100-pct:.0f}%)")

print(f"\nFull per-stat breakdown:")
print(outlier_df.to_string(index=False))

# ============================================================
# 6. SUMMARY & DECISION HELPER
# ============================================================
print(f"\n{'='*70}")
print(f"6. IMPACT SUMMARY — What happens if we NaN stale stats?")
print(f"{'='*70}\n")

# Option A: NaN ALL stale stats
print(f"Option A — NaN ALL stats older than {STALE_MONTHS} months:")
remaining_a = total_vals - total_stale
print(f"  Stat values removed: {total_stale:,} / {total_vals:,} ({100*total_stale/total_vals:.1f}%)")
print(f"  Stat values remaining: {remaining_a:,}")

# How many players would lose ALL their stats?
all_stale = (stale_counts_per_player == stat_counts_per_player) & (stat_counts_per_player > 0)
print(f"  Players losing ALL stats: {all_stale.sum():,} / {len(of_model_recent):,}")

# Option B: NaN only stale OUTLIER stats
total_stale_outliers = outlier_df["stale_outliers"].sum()
print(f"\nOption B — NaN only stale outliers (±{OUTLIER_STD_DEV} std + >{STALE_MONTHS}mo):")
print(f"  Stat values removed: {total_stale_outliers:,} / {total_vals:,} ({100*total_stale_outliers/total_vals:.1f}%)")
print(f"  More surgical, but leaves non-outlier stale data in place")

# Clean up internal columns from of_model_recent (keep months_before_grad cols for downstream use)
internal_cols = [c for c in of_model_recent.columns if c.startswith("_")]
of_model_recent.drop(columns=internal_cols, inplace=True)
print(f"\n✓ of_model_recent is ready ({of_model_recent.shape})")

Parsed 13 stat date columns.

1. PER-STAT STALENESS  (stale = >24 months before graduation)

          stat  has_value  has_date  no_date  stale  stale_pct
 exit_velo_max       1777      1777        0    424       23.9
 exit_velo_avg       1567      1567        0    391       25.0
  distance_max       1567      1567        0    379       24.2
  sweet_spot_p       1563      1563        0    625       40.0
hand_speed_max       1475      1475        0    511       34.6
 bat_speed_max       1475      1475        0    416       28.2
   rot_acc_max       1475      1475        0    494       33.5
    hard_hit_p       1365      1365        0    264       19.3
    sixty_time       1712      1712        0    476       27.8
   thirty_time        828       827        1    113       13.6
 ten_yard_time        858       857        1    155       18.1
 run_speed_max        901       900        1    134       14.9
       of_velo       1697      1697        0    457       26.9

Overall: 4,839 / 18,260 

/var/folders/xp/hjmh03ws3nd2zyhxj5kw6j7w0000gn/T/ipykernel_38714/2616661880.py:161: RuntimeWarning: invalid value encountered in scalar divide
  print(f"  {group:>12}: {n_affected:>4,} / {n_total:>5,} players have ≥1 stale stat ({100*n_affected/n_total:.1f}%)")


In [33]:
# ============================================================
# APPLY OPTION B — NaN only stale outliers (MULTI-PASS)
# ============================================================
# Each pass NaN's stale outliers, then the mean/std shift,
# revealing new outliers. Repeat until no more are found.

MAX_PASSES = 5
grand_total = 0

for pass_i in range(1, MAX_PASSES + 1):
    pass_total = 0
    pass_log = []

    for stat_col, date_col in STAT_DATE_PAIRS:
        months_col = f"{stat_col}__months_before_grad"
        stat_nand = 0

        for group in ["P4", "Non P4 D1"]:
            mask = of_model_recent["committment_group"] == group
            vals = of_model_recent.loc[mask, stat_col]
            mean = vals.mean()
            std  = vals.std()

            if pd.isna(mean) or pd.isna(std) or std == 0:
                continue

            is_outlier = (vals < mean - OUTLIER_STD_DEV * std) | (vals > mean + OUTLIER_STD_DEV * std)
            is_stale   = of_model_recent.loc[mask, months_col] > STALE_MONTHS

            to_nan = is_outlier & is_stale
            n = to_nan.sum()

            if n > 0:
                of_model_recent.loc[to_nan[to_nan].index, stat_col] = np.nan
                of_model_recent.loc[to_nan[to_nan].index, date_col] = np.nan
                stat_nand += n

        if stat_nand > 0:
            pass_log.append({"stat": stat_col, "values_removed": stat_nand})
            pass_total += stat_nand

    grand_total += pass_total
    print(f"Pass {pass_i}: NaN'd {pass_total} stale outlier values")
    if pass_log:
        for row in pass_log:
            print(f"    {row['stat']:>15}: {row['values_removed']}")

    if pass_total == 0:
        print(f"  No more stale outliers found — stopping.")
        break

print(f"\nTotal across {pass_i} passes: {grand_total} values NaN'd")
print(f"  (±{OUTLIER_STD_DEV} std dev + >{STALE_MONTHS} months before graduation)")

# --- Verify ---
remaining = 0
for stat_col, _ in STAT_DATE_PAIRS:
    months_col = f"{stat_col}__months_before_grad"
    for group in ["P4", "Non P4 D1"]:
        mask = of_model_recent["committment_group"] == group
        vals = of_model_recent.loc[mask, stat_col]
        mean = vals.mean()
        std  = vals.std()
        if pd.isna(mean) or pd.isna(std) or std == 0:
            continue
        is_outlier = (vals < mean - OUTLIER_STD_DEV * std) | (vals > mean + OUTLIER_STD_DEV * std)
        is_stale   = of_model_recent.loc[mask, months_col] > STALE_MONTHS
        remaining += (is_outlier & is_stale).sum()

print(f"Stale outliers remaining: {remaining}")
print(f"of_model_recent shape: {of_model_recent.shape}")

Pass 1: NaN'd 372 stale outlier values
      exit_velo_max: 53
      exit_velo_avg: 41
       distance_max: 39
       sweet_spot_p: 31
     hand_speed_max: 24
      bat_speed_max: 35
        rot_acc_max: 17
         hard_hit_p: 11
         sixty_time: 41
        thirty_time: 11
      ten_yard_time: 8
      run_speed_max: 12
            of_velo: 49
Pass 2: NaN'd 133 stale outlier values
      exit_velo_max: 35
      exit_velo_avg: 16
       distance_max: 14
     hand_speed_max: 6
      bat_speed_max: 7
        rot_acc_max: 3
         hard_hit_p: 6
         sixty_time: 17
        thirty_time: 5
      ten_yard_time: 1
      run_speed_max: 4
            of_velo: 19
Pass 3: NaN'd 25 stale outlier values
      exit_velo_max: 6
      exit_velo_avg: 3
       distance_max: 5
     hand_speed_max: 1
         hard_hit_p: 3
         sixty_time: 5
      run_speed_max: 1
            of_velo: 1
Pass 4: NaN'd 11 stale outlier values
      exit_velo_max: 6
         sixty_time: 5
Pass 5: NaN'd 2 stale ou

## Logistic Regression Baseline

In [34]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, confusion_matrix
import matplotlib.pyplot as plt

FEATURES = [
    "height", "weight",
    "exit_velo_max", "distance_max", "bat_speed_max",
    "sixty_time", "of_velo",
]

X = of_model_recent[FEATURES]
y = of_model_recent["p4_or_not"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

pipe = Pipeline([
    ("impute", KNNImputer(n_neighbors=10)),
    ("scale", StandardScaler()),
    ("lr", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
])

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

print(classification_report(y_test, y_pred, target_names=["Non P4 D1", "P4"]))
print(confusion_matrix(y_test, y_pred), "\n")

coefs = pd.Series(pipe["lr"].coef_[0], index=FEATURES).sort_values()
print(coefs)

              precision    recall  f1-score   support

   Non P4 D1       0.86      0.60      0.71       346
          P4       0.31      0.64      0.42        98

    accuracy                           0.61       444
   macro avg       0.58      0.62      0.56       444
weighted avg       0.74      0.61      0.64       444

[[208 138]
 [ 35  63]] 

bat_speed_max   -0.256028
sixty_time      -0.231173
weight           0.085621
of_velo          0.119239
height           0.168185
distance_max     0.300832
exit_velo_max    0.310150
dtype: float64


## Generate Out-of-Fold D1 Probabilities

We need `d1_prob` as a meta-feature for the P4 model. Can't use the saved D1 model directly because these D1 players were in its training data — that would leak. Instead, retrain the D1 model on the **full dataset** (including Non-D1) with cross_val_predict to get leak-free calibrated probabilities for every player, then filter to just the D1 subset.

In [35]:
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.calibration import CalibratedClassifierCV
import numpy as np

# Load the FULL outfielder dataset (including Non-D1) for D1 model training
hitters_full = pd.read_csv(Path('hitters_cleaned.csv'), low_memory=False)
ofs_full = hitters_full[hitters_full['resolved_position'] == 'OF'].copy()
ofs_full = ofs_full[ofs_full['committment_group'] != 'Unknown']
ofs_full['d1_or_not'] = (ofs_full['committment_group'] != 'Non D1').astype(int)

FEATURES_D1 = ['height', 'weight', 'exit_velo_max', 'distance_max', 'sixty_time', 'of_velo', 'bat_speed_max']

X_full = ofs_full[FEATURES_D1]
y_full = ofs_full['d1_or_not']
neg, pos = (y_full == 0).sum(), (y_full == 1).sum()

print(f'Full OF dataset: {len(ofs_full)} players ({pos} D1, {neg} Non-D1)')

# Out-of-fold D1 probabilities using CalibratedClassifierCV inside each fold
# cross_val_predict with method='predict_proba' gives leak-free probabilities
xgb_d1 = XGBClassifier(
    n_estimators=500, max_depth=4, learning_rate=0.01,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=1.0, reg_lambda=3.0, min_child_weight=5,
    scale_pos_weight=neg / pos, eval_metric='logloss',
    random_state=42, enable_categorical=False,
)
cal_d1 = CalibratedClassifierCV(xgb_d1, method='isotonic', cv=3)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_probs = cross_val_predict(cal_d1, X_full, y_full, cv=skf, method='predict_proba')[:, 1]

# Attach to full dataset
ofs_full['d1_prob'] = oof_probs

# Filter to D1 players only and merge into of_model_recent
d1_players = ofs_full[ofs_full['d1_or_not'] == 1][['d1_prob']]
of_model_recent = of_model_recent.join(d1_players, how='left')

print(f'\nD1 prob stats for D1 players (n={of_model_recent["d1_prob"].notna().sum()}):')
print(f'  mean: {of_model_recent["d1_prob"].mean():.3f}')
print(f'  std:  {of_model_recent["d1_prob"].std():.3f}')
print(f'  min:  {of_model_recent["d1_prob"].min():.3f}')
print(f'  max:  {of_model_recent["d1_prob"].max():.3f}')

print(f'\nD1 prob by group:')
for group in ['P4', 'Non P4 D1']:
    mask = of_model_recent['committment_group'] == group
    vals = of_model_recent.loc[mask, 'd1_prob']
    print(f'  {group:>12}: mean={vals.mean():.3f}  std={vals.std():.3f}  median={vals.median():.3f}')

Full OF dataset: 8516 players (2217 D1, 6299 Non-D1)

D1 prob stats for D1 players (n=2217):
  mean: 0.368
  std:  0.179
  min:  0.000
  max:  0.942

D1 prob by group:
            P4: mean=0.438  std=0.185  median=0.439
     Non P4 D1: mean=0.348  std=0.172  median=0.328


## XGBoost — P4 vs Non-P4 D1 (7 core + d1_prob)

Using the 7 production features plus the D1 probability as a meta-feature.

In [48]:
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import brier_score_loss

FEATURES_P4 = ['height', 'weight', 'exit_velo_max', 'distance_max', 'sixty_time', 'of_velo', 'bat_speed_max', 'd1_prob']

X = of_model_recent[FEATURES_P4]
y = of_model_recent['p4_or_not']
neg, pos = (y == 0).sum(), (y == 1).sum()
print(f'Class balance: {neg} Non-P4 D1, {pos} P4  (ratio {neg/pos:.2f}:1)\n')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

xgb_p4 = XGBClassifier(
    n_estimators=250, max_depth=3, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=1.5, reg_lambda=5.0, min_child_weight=10,
    scale_pos_weight=neg / pos, eval_metric='logloss',
    random_state=42, enable_categorical=False,
)

xgb_p4.fit(X_train, y_train)
y_pred = xgb_p4.predict(X_test)
y_proba = xgb_p4.predict_proba(X_test)[:, 1]

print(f'XGBoost — {len(FEATURES_P4)} Features (7 core + d1_prob)')
print('=' * 60)
print(classification_report(y_test, y_pred, target_names=['Non P4 D1', 'P4']))
print(confusion_matrix(y_test, y_pred))

auc = roc_auc_score(y_test, y_proba)
print(f'\nAUC-ROC: {auc:.4f}')

importances = pd.Series(xgb_p4.feature_importances_, index=FEATURES_P4).sort_values(ascending=False)
print(f'\nFeature importance:')
print(importances.to_string())

# Stratified CV — use 5 folds, min ~98 P4 players means ~20 per fold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv = cross_validate(
    xgb_p4, X, y, cv=skf,
    scoring=['roc_auc', 'f1', 'precision', 'recall', 'accuracy'],
    return_train_score=True,
)

print(f'\nStratified 5-Fold CV:')
print('=' * 60)
for metric in ['roc_auc', 'accuracy', 'f1', 'precision', 'recall']:
    tr = cv[f'train_{metric}']
    te = cv[f'test_{metric}']
    label = 'AUC-ROC' if metric == 'roc_auc' else metric
    print(f'  {label:>12}:  train {tr.mean():.3f} (+/- {tr.std():.3f})  |  test {te.mean():.3f} (+/- {te.std():.3f})')
print(f'\n  Overfit gap (AUC): {cv["train_roc_auc"].mean() - cv["test_roc_auc"].mean():.3f}')

# Calibration
xgb_p4_cal = CalibratedClassifierCV(xgb_p4, method='isotonic', cv=5)
xgb_p4_cal.fit(X_train, y_train)
y_proba_cal = xgb_p4_cal.predict_proba(X_test)[:, 1]
y_pred_cal = (y_proba_cal >= 0.32).astype(int)

fraction_cal, mean_cal = calibration_curve(y_test, y_proba_cal, n_bins=10)
brier_raw = brier_score_loss(y_test, y_proba)
brier_cal = brier_score_loss(y_test, y_proba_cal)

print(f'\nCalibrated model:')
print('=' * 60)
print(classification_report(y_test, y_pred_cal, target_names=['Non P4 D1', 'P4']))
print(confusion_matrix(y_test, y_pred_cal))
print(f'Brier:  raw = {brier_raw:.4f}  |  calibrated = {brier_cal:.4f}')
print(f'\nCalibration curve (calibrated, 8 bins):')
print(f'{"Predicted":>12}  {"Actual":>8}  {"Gap":>8}')
for pred, actual in zip(mean_cal, fraction_cal):
    gap = pred - actual
    flag = '  <- off' if abs(gap) > 0.10 else ''
    print(f'  {pred:>10.2f}  {actual:>8.2f}  {gap:>+8.2f}{flag}')

# Comparison
print(f'\n{"=" * 60}')
print('COMPARISON: LogReg baseline vs XGBoost + d1_prob')
print(f'{"=" * 60}')
rpt = classification_report(y_test, y_pred, target_names=['Non P4 D1', 'P4'], output_dict=True)
print(f'  LogReg (7 feat):       P4 prec=0.31  P4 recall=0.64  P4 F1=0.42  acc=0.61')
print(f'  XGBoost (8 feat):      P4 prec={rpt["P4"]["precision"]:.2f}  P4 recall={rpt["P4"]["recall"]:.2f}  P4 F1={rpt["P4"]["f1-score"]:.2f}  acc={rpt["accuracy"]:.2f}  AUC={auc:.3f}')

Class balance: 1727 Non-P4 D1, 490 P4  (ratio 3.52:1)

XGBoost — 8 Features (7 core + d1_prob)
              precision    recall  f1-score   support

   Non P4 D1       0.86      0.67      0.75       346
          P4       0.34      0.61      0.44        98

    accuracy                           0.66       444
   macro avg       0.60      0.64      0.60       444
weighted avg       0.74      0.66      0.68       444

[[231 115]
 [ 38  60]]

AUC-ROC: 0.7089

Feature importance:
exit_velo_max    0.157150
d1_prob          0.139502
distance_max     0.137586
sixty_time       0.135319
weight           0.120035
of_velo          0.109174
bat_speed_max    0.101679
height           0.099556

Stratified 5-Fold CV:
       AUC-ROC:  train 0.850 (+/- 0.007)  |  test 0.683 (+/- 0.038)
      accuracy:  train 0.738 (+/- 0.010)  |  test 0.652 (+/- 0.021)
            f1:  train 0.576 (+/- 0.011)  |  test 0.427 (+/- 0.031)
     precision:  train 0.448 (+/- 0.011)  |  test 0.336 (+/- 0.025)
        recall

## Save Production P4 Model

Save the calibrated XGBoost P4 model (8 features, threshold=0.50) to the model directory.

In [50]:
import joblib
import json
import os
from datetime import datetime

# ============================================================
# Save calibrated XGBoost P4 model to production model directory
# ============================================================
THRESHOLD = 0.32
VERSION = f"version_{datetime.now().strftime('%m%d%Y')}"
SAVE_DIR = os.path.abspath(os.path.join(
    os.getcwd(), "..", "..", "..", "models", "models_of", "models_p4_or_not_of", VERSION
))
os.makedirs(SAVE_DIR, exist_ok=True)

# --- Train final model on full training set ---
FEATURES_P4 = ["height", "weight", "exit_velo_max", "distance_max", "sixty_time", "of_velo", "bat_speed_max", "d1_prob"]

X = of_model_recent[FEATURES_P4]
y = of_model_recent["p4_or_not"]
neg, pos = (y == 0).sum(), (y == 1).sum()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

xgb_final = XGBClassifier(
    n_estimators=300, max_depth=3, learning_rate=0.03,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=1.5, reg_lambda=5.0, min_child_weight=8,
    scale_pos_weight=neg / pos, eval_metric="logloss",
    random_state=42, enable_categorical=False,
)
xgb_final.fit(X_train, y_train)

xgb_final_cal = CalibratedClassifierCV(xgb_final, method="isotonic", cv=5)
xgb_final_cal.fit(X_train, y_train)

# --- Eval metrics on holdout ---
y_proba_test = xgb_final_cal.predict_proba(X_test)[:, 1]
y_pred_test = (y_proba_test >= THRESHOLD).astype(int)

auc = roc_auc_score(y_test, y_proba_test)
brier = brier_score_loss(y_test, y_proba_test)
report = classification_report(y_test, y_pred_test, target_names=["Non P4 D1", "P4"], output_dict=True)
fraction_cal, mean_cal = calibration_curve(y_test, y_proba_test, n_bins=8)
max_cal_gap = max(abs(m - f) for m, f in zip(mean_cal, fraction_cal))

# --- Save model ---
joblib.dump(xgb_final_cal, os.path.join(SAVE_DIR, "calibrated_xgb_model.pkl"))
print(f"Saved: calibrated_xgb_model.pkl")

# --- Save model config ---
model_config = {
    "model_version": f"of_p4_xgb_cal_{VERSION}",
    "creation_date": datetime.now().isoformat(),
    "model_type": "calibrated_xgboost_outfielder_p4",
    "calibration_method": "isotonic",
    "threshold": THRESHOLD,
    "features": FEATURES_P4,
    "requires_d1_prob": True,
    "hyperparameters": {
        "n_estimators": 300,
        "max_depth": 3,
        "learning_rate": 0.03,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_alpha": 1.5,
        "reg_lambda": 5.0,
        "min_child_weight": 8,
        "scale_pos_weight": round(neg / pos, 4),
    },
    "performance_metrics": {
        "auc_roc": round(auc, 4),
        "brier_score": round(brier, 4),
        "max_calibration_gap": round(max_cal_gap, 4),
        "p4_precision": round(report["P4"]["precision"], 4),
        "p4_recall": round(report["P4"]["recall"], 4),
        "p4_f1": round(report["P4"]["f1-score"], 4),
        "accuracy": round(report["accuracy"], 4),
        "threshold_used": THRESHOLD,
    },
    "dataset_info": {
        "total_samples": len(of_model_recent),
        "train_samples": len(X_train),
        "test_samples": len(X_test),
        "p4_rate": round(pos / (neg + pos), 4),
        "d1_only": True,
        "stale_cleaning": "Option B multi-pass (stale outliers only, +/-2 std + >24mo)",
    },
    "calibration_curve": {
        "predicted": [round(float(m), 4) for m in mean_cal],
        "actual": [round(float(f), 4) for f in fraction_cal],
    },
}

with open(os.path.join(SAVE_DIR, "model_config.json"), "w") as f:
    json.dump(model_config, f, indent=2)
print(f"Saved: model_config.json")

# --- Save feature metadata ---
feature_metadata = {
    "features": FEATURES_P4,
    "required_columns": ["height", "weight", "exit_velo_max", "distance_max", "sixty_time", "of_velo", "bat_speed_max"],
    "meta_features": ["d1_prob"],
    "notes": "d1_prob comes from the calibrated D1 model. XGBoost handles NaN natively. No imputation or scaling needed.",
}

with open(os.path.join(SAVE_DIR, "feature_metadata.json"), "w") as f:
    json.dump(feature_metadata, f, indent=2)
print(f"Saved: feature_metadata.json")

# --- Print summary ---
print(f"\nModel saved to: {SAVE_DIR}")
print(f"\n{'=' * 60}")
print(f"PRODUCTION MODEL METRICS (threshold={THRESHOLD})")
print(f"{'=' * 60}")
print(f"  AUC-ROC:           {auc:.4f}")
print(f"  Brier score:       {brier:.4f}")
print(f"  Max cal gap:       {max_cal_gap:.4f}")
print(f"  P4 Precision:      {report['P4']['precision']:.4f}")
print(f"  P4 Recall:         {report['P4']['recall']:.4f}")
print(f"  P4 F1:             {report['P4']['f1-score']:.4f}")
print(f"  Accuracy:          {report['accuracy']:.4f}")
print(f"  Features:          {len(FEATURES_P4)} ({', '.join(FEATURES_P4)})")
print(f"  Threshold:         {THRESHOLD}")
print(f"  Training samples:  {len(X_train)}")
print(f"  Test samples:      {len(X_test)}")
print(f"  P4 base rate:      {pos/(neg+pos):.1%}")

print(f"\nCalibration curve:")
print(f"  {'Predicted':>10}  {'Actual':>8}  {'Gap':>8}")
for pred, actual in zip(mean_cal, fraction_cal):
    gap = pred - actual
    flag = "  <- off" if abs(gap) > 0.10 else ""
    print(f"  {pred:>10.2f}  {actual:>8.2f}  {gap:>+8.2f}{flag}")

print(f"\nFiles in {VERSION}/:")
for f in sorted(os.listdir(SAVE_DIR)):
    size = os.path.getsize(os.path.join(SAVE_DIR, f))
    print(f"  {f:<35} {size:>8,} bytes")

Saved: calibrated_xgb_model.pkl
Saved: model_config.json
Saved: feature_metadata.json

Model saved to: /Users/ryankolodziejczyk/Documents/AI Baseball Recruitment/code/backend/ml/models/models_of/models_p4_or_not_of/version_04202026

PRODUCTION MODEL METRICS (threshold=0.32)
  AUC-ROC:           0.7016
  Brier score:       0.1568
  Max cal gap:       0.2460
  P4 Precision:      0.4366
  P4 Recall:         0.3163
  P4 F1:             0.3669
  Accuracy:          0.7590
  Features:          8 (height, weight, exit_velo_max, distance_max, sixty_time, of_velo, bat_speed_max, d1_prob)
  Threshold:         0.32
  Training samples:  1773
  Test samples:      444
  P4 base rate:      22.1%

Calibration curve:
   Predicted    Actual       Gap
        0.08      0.08     -0.00
        0.19      0.18     +0.01
        0.29      0.33     -0.04
        0.43      0.37     +0.06
        0.55      0.50     +0.05
        0.67      0.67     +0.00
        0.75      1.00     -0.25  <- off

Files in version_0